## 1. Importing Dataset 



In [16]:
import pandas as pd
import numpy as np


RAW_PATH = "data/raw/real_estate_master.csv"
OUT_PATH = "data/processed/cleaned_listings.csv"

df = pd.read_csv(RAW_PATH)
df.info()
#df.describe(include='all')
#df.isnull().sum().sort_values(ascending=False)
# df.duplicated().sum()
print("Shape:", df.shape)
#df.head()
#print(df.shape)
#print(df["city"].unique())
#print(df["property_type"].unique())

<class 'pandas.DataFrame'>
RangeIndex: 13828 entries, 0 to 13827
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   location               13814 non-null  str    
 1   area                   13814 non-null  float64
 2   price                  13815 non-null  float64
 3   price_currency         13815 non-null  str    
 4   status                 4690 non-null   float64
 5   new/resale             13814 non-null  float64
 6   price_negotiable       13814 non-null  float64
 7   description            13264 non-null  str    
 8   security_deposit       13814 non-null  float64
 9   facing                 11807 non-null  str    
 10  furnished              13814 non-null  float64
 11  age of property        13828 non-null  int64  
 12  Lift(s)                13814 non-null  float64
 13  Full Power Backup      13814 non-null  float64
 14  24 X 7 Security        13814 non-null  float64
 15  Children's pl

2. Droping Dupliicates

In [17]:
# print("Duplicates:", df.duplicated().sum())
# df["price"].describe()
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate rows after droping:", df.duplicated().sum())
print (df.shape)

Duplicate rows: 924
Duplicate rows after droping: 0
(12904, 41)


3. Missing Values

In [18]:
missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

missing_table = pd.DataFrame({
    "missing": missing,
    "missing_pct": missing_pct
})

print(missing_table.sort_values("missing_pct", ascending=False))
print(df.shape)
drop_cols = missing_pct[missing_pct > 75]

print("Columns being removed:")
#print(drop_cols)
df = df.drop(columns=drop_cols.index)


print (df.shape)

                       missing  missing_pct
project_score            12430    96.326720
Golf Course              12339    95.621513
Cafeteria                12045    93.343149
Multipurpose Room        11818    91.584005
builder_experience       11707    90.723807
Indoor Games             11670    90.437074
Staff Quarter            11634    90.158091
Maintenance Staff        11366    88.081215
Shopping Mall            11213    86.895536
ATM                      11203    86.818041
Rain Water Harvesting    11162    86.500310
Hospital                 11104    86.050837
Vaastu Compliant         10925    84.663670
School                   10724    83.106014
Intercom                 10296    79.789213
Car Parking               9339    72.372908
locality_score            8833    68.451643
status                    8510    65.948543
facing                    1950    15.111593
description                442     3.425294
security_deposit             4     0.030998
area                         4  

## 4. Dropping Unparseable Numeric data

In [19]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["area"] = pd.to_numeric(df["area"], errors="coerce")
df["security_deposit"] = pd.to_numeric(df["security_deposit"], errors="coerce")

# drop rows where price or area could not be parsed / are missing
before = len(df)
df = df.dropna(subset=["price", "area"])
after = len(df)
print(f"Dropped {before - after} rows with unparseable price/area")


Dropped 4 rows with unparseable price/area


In [20]:
print(df["Car Parking"].value_counts(dropna=False))
print(df["locality_score"].value_counts(dropna=False))
print(df["status"].value_counts(dropna=False))
print (df["facing"].value_counts(dropna=False))
print (df.shape)

Car Parking
NaN    9335
1.0    3565
Name: count, dtype: int64
locality_score
NaN     8829
9.3     1294
8.0      470
9.5      340
7.4      261
6.3      249
8.1      141
9.1      118
7.1      108
9.4      101
9.2       90
7.9       83
8.4       79
5.3       79
8.2       68
8.9       60
8.8       56
7.2       55
8.5       48
6.8       42
6.2       40
8.6       38
10.0      37
9.7       30
9.8       22
9.0       22
7.3       20
8.3       20
7.7       14
9.6       10
6.0       10
9.9        8
4.9        7
6.4        6
8.7        5
5.8        5
7.6        5
4.4        5
6.7        4
5.2        4
6.5        4
6.9        4
7.5        3
5.7        2
5.0        2
6.6        1
4.7        1
Name: count, dtype: int64
status
NaN    8506
1.0    3964
0.0     430
Name: count, dtype: int64
facing
east         6300
northeast    2735
NaN          1946
west          717
north         525
northwest     317
south         161
southeast     136
southwest      63
Name: count, dtype: int64
(12900, 26)


In [21]:
df["Car Parking_missing"] = df["Car Parking"].isna().astype(int)
df = df.drop(columns=["Car Parking"])
df["status"] = df["status"].fillna("Unknown").astype(str) # then one-hot encode later: status_0.0, status_1.0, status_Unknown
df["facing"] = df["facing"].fillna("Unknown").astype(str)
if "description" in df.columns:
    df["description"] = df["description"].fillna("")
print(df["locality_score"].isna().sum())
df["locality_score_missing"] = df["locality_score"].isna().astype(int)
df["locality_score"] = df.groupby("city")["locality_score"].transform(
    lambda x: x.fillna(x.median())
)

print(df["locality_score_missing"].value_counts())
print (df.shape)

8829
locality_score_missing
1    8829
0    4071
Name: count, dtype: int64
(12900, 27)


In [22]:
print(
    df.groupby("city")["locality_score"]
      .agg(["count", "median"])
      .sort_values("count")
)

            count  median
city                     
Chandigarh   1940     7.1
Lucknow      2036     6.3
Pune         2701     8.0
Ghaziabad    6223     9.3


In [23]:
remaining_nulls = df.isna().sum()
remaining_nulls[remaining_nulls > 0]
#df.isna().sum()
remaining_nulls[remaining_nulls > 0]

Series([], dtype: int64)

In [24]:
print("Area <= 0:", (df["area"] <= 0).sum())

print("Price <= 0:", (df["price"] <= 0).sum())

print("Age < 0:", (df["age of property"] < 0).sum())

print("Age > 200:", (df["age of property"] > 200).sum())

print("Security deposit < 0:", (df["security_deposit"] < 0).sum())

Area <= 0: 0
Price <= 0: 0
Age < 0: 0
Age > 200: 0
Security deposit < 0: 0


In [25]:
print("Area < 100:", (df["area"] < 100).sum())
print("Area > 20000:", (df["area"] > 20000).sum())

print("Price < 50,000:", (df["price"] < 50000).sum())
print("Price > 50 crore:", (df["price"] > 5e8).sum())

print("Age > 60:", (df["age of property"] > 60).sum())

Area < 100: 6
Area > 20000: 4
Price < 50,000: 1
Price > 50 crore: 0
Age > 60: 70


In [26]:
# Sanity bounds — drop clearly erroneous outliers, not just <=0 values
before = len(df)
df = df[(df["area"] >= 100) & (df["area"] <= 20000)]
df = df[(df["price"] >= 50000) & (df["price"] <= 5e8)]
df = df[df["age of property"] <= 60]
print(f"Dropped {before - len(df)} rows as unrealistic outliers")
print(df.shape)

Dropped 80 rows as unrealistic outliers
(12820, 27)


In [27]:
amenity_cols= [
    "Lift(s)", "Full Power Backup", "24 X 7 Security", "Children's play area",
    "Club House", "Gymnasium", "Swimming Pool", "Sports Facility",
    "Jogging Track", "Landscaped Gardens"
]
#amenity_cols = [c for c in amenity_cols if c in df.columns]

for col in amenity_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print(df[col].value_counts(dropna=False).head(20))


for col in amenity_cols:
    df[col] = df[col].fillna(0)




--- Lift(s) ---
Lift(s)
0.0    8919
1.0    3901
Name: count, dtype: int64
Lift(s)
0.0    8919
1.0    3901
Name: count, dtype: int64

--- Full Power Backup ---
Full Power Backup
0.0    8572
1.0    4248
Name: count, dtype: int64
Full Power Backup
0.0    8572
1.0    4248
Name: count, dtype: int64

--- 24 X 7 Security ---
24 X 7 Security
0.0    9187
1.0    3633
Name: count, dtype: int64
24 X 7 Security
0.0    9187
1.0    3633
Name: count, dtype: int64

--- Children's play area ---
Children's play area
0.0    6908
1.0    5912
Name: count, dtype: int64
Children's play area
0.0    6908
1.0    5912
Name: count, dtype: int64

--- Club House ---
Club House
0.0    10377
1.0     2443
Name: count, dtype: int64
Club House
0.0    10377
1.0     2443
Name: count, dtype: int64

--- Gymnasium ---
Gymnasium
0.0    8344
1.0    4476
Name: count, dtype: int64
Gymnasium
0.0    8344
1.0    4476
Name: count, dtype: int64

--- Swimming Pool ---
Swimming Pool
0.0    9692
1.0    3128
Name: count, dtype: int64
Swi

In [37]:
# Check inconsistent numeric formats

print("=== NUMERIC FORMAT CHECK ===")

for col in df.columns:
    if df[col].dtype == "object" or str(df[col].dtype) == "string":
        
        values = df[col].dropna().astype("string").str.strip()

        # Values containing numbers mixed with non-numeric characters
        numeric_like = values.str.contains(r"\d", regex=True)
        mixed_format = values[numeric_like & ~values.str.fullmatch(r"[-+]?\d*\.?\d+")]

        if not mixed_format.empty:
            print(f"\n--- {col} ---")
            print(f"Potential mixed formats: {len(mixed_format)}")
            print(mixed_format.unique()[:15])

=== NUMERIC FORMAT CHECK ===


In [38]:
# Check inconsistent missing-value representations

print("=== MISSING VALUE FORMAT CHECK ===")

missing_placeholders = ["?", "N/A", "NA", "na", "null", "None", "-", "--"]

found = False

for col in df.columns:
    values = df[col].astype("string").str.strip()
    matches = values[values.isin(missing_placeholders)]

    if not matches.empty:
        found = True
        print(f"\n--- {col} ---")
        print(f"Placeholder values: {len(matches)}")
        print(matches.value_counts())

if not found:
    print("No inconsistent missing-value placeholders found.")

=== MISSING VALUE FORMAT CHECK ===
No inconsistent missing-value placeholders found.


In [39]:
# Check boolean / binary format consistency

print("=== BOOLEAN FORMAT CHECK ===")

binary_candidates = [
    "car_parking",
    "loan_required",
    "site_visit",
    "negotiation_done",
    "booking_done"
]

for col in binary_candidates:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False))

=== BOOLEAN FORMAT CHECK ===


In [43]:
# Final inconsistent-format summary

print("=== FORMAT CONSISTENCY SUMMARY ===")

issues = []

for col in cat_cols:
    original = df[col].astype("string")
    stripped = original.str.strip()

    # Whitespace
    whitespace_count = original.ne(stripped).sum()

    # Case / spelling variants
    normalized = stripped.str.lower()

    variants = (
        pd.DataFrame({
            "original": stripped,
            "normalized": normalized
        })
        .dropna()
        .groupby("normalized")["original"]
        .unique()
    )

    case_variants = variants[variants.apply(len) > 1]

    if whitespace_count > 0 or not case_variants.empty:
        issues.append({
            "column": col,
            "whitespace_issues": whitespace_count,
            "case_or_label_variants": len(case_variants)
        })

if issues:
    format_summary = pd.DataFrame(issues)
    print(format_summary.to_string(index=False))
else:
    print("No categorical format inconsistencies found.")
    
df["location"] = df["location"].astype("string").str.strip().str.title()   
df["description"] = df["description"].astype("string").str.strip()
print("Remaining location whitespace:",
      df["location"].astype("string").ne(
          df["location"].astype("string").str.strip()
      ).sum())

print("Remaining description whitespace:",
      df["description"].astype("string").ne(
          df["description"].astype("string").str.strip()
      ).sum())


=== FORMAT CONSISTENCY SUMMARY ===
     column  whitespace_issues  case_or_label_variants
   location                  2                       1
description              10941                       0
Remaining location whitespace: 0
Remaining description whitespace: 0


feature block

In [41]:
df["amenity_count"] = df[amenity_cols].sum(axis=1)
df["price_per_sqft"] = df["price"] / df["area"]
df[["price", "area", "price_per_sqft", "amenity_count"]].describe()

,price,area,price_per_sqft,amenity_count
count,1.282000e+04,12820.000000,12820.000000,12820.000000
mean,7.439025e+06,1543.812480,4162.073328,2.726287
std,1.216035e+07,1233.273505,3402.061754,2.835294
min,1.000000e+05,100.000000,100.000000,0.000000
25%,1.652750e+06,1000.000000,1600.000000,0.000000
50%,4.415500e+06,1205.000000,3840.000000,2.000000
75%,7.900000e+06,1743.000000,5448.645783,5.000000
max,3.397680e+08,20000.000000,50000.000000,10.000000


In [42]:
print("Final shape:", df.shape)
print("\nRows per city:")
print(df["city"].value_counts())
print("\nRows per property_type:")
print(df["property_type"].value_counts())

df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")

Final shape: (12820, 29)

Rows per city:
city
Ghaziabad     6192
Pune          2685
Lucknow       2035
Chandigarh    1908
Name: count, dtype: int64

Rows per property_type:
property_type
builderfloor    6411
plot            4870
villa           1539
Name: count, dtype: int64

Saved cleaned dataset to data/processed/cleaned_listings.csv
